# Module 8 Assignment: Give Your Persona Agent Memory With Sessions

Lab 7 added a `SQLiteSession` to Study Buddy: first in memory, to fix two `Runner.run` calls that
did not remember each other, then file backed, so the conversation survives a kernel restart. The
Gradio `chat()` function got simpler in the process, no more rebuilding `history` by hand.

This assignment makes you do the same thing to your **own** Persona Agent, the one you rebuilt on
the OpenAI Agents SDK in the Module 7 assignment. Nothing about your agent's persona, notes, or
three tools needs to change here: you are giving it memory, not redesigning it.

**What you'll build:**
1. Your Module 7 assignment's Agent, Azure configuration, and three tools, carried over unchanged
2. A quick demonstration that two separate `Runner.run` calls do not remember each other
3. A file backed `SQLiteSession` that fixes it
4. A simplified Gradio `chat()` function that uses `session=session` instead of rebuilding `history`

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you get stuck on any step, look at how Lab 7 made the same kind of change to Study Buddy, and do
the analogous thing to your own agent.

## 0. Install the packages (run once)

Same as Lab 7: no new package beyond what your Module 7 assignment already installed.
`SQLiteSession` ships with `openai-agents` itself.

In [26]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

Resolved 76 packages in 300ms                                        
Checked 76 packages in 2ms


## 1. Imports

**TODO:** Copy the imports from your Module 7 assignment, plus one new name: `SQLiteSession`,
imported from `agents` alongside `Agent` and `Runner`, the same way Lab 7, Step 1 added it.

In [27]:
# TODO: copy your imports from the Module 7 assignment, plus SQLiteSession from agents
# TODO: copy your imports from Lab 6 (including gradio as gr), plus requests and anything your tools need
from dotenv import load_dotenv
import os
import asyncio
import html
from typing import Literal, Optional

import requests
from openai import AsyncAzureOpenAI
from agents import (
    Agent,
    Runner,
    function_tool,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
    SQLiteSession,
)
from pypdf import PdfReader
import gradio as gr

## 2. Load your Azure OpenAI credentials from `.env`

Identical to the Module 7 assignment. Sessions do not change which provider you talk to.

In [28]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Endpoint loaded: True
API key loaded: True


## 3. Configure the OpenAI Agents SDK for Azure OpenAI

**TODO:** Copy this setup from your Module 7 assignment, unchanged: `set_tracing_disabled(True)`,
your `AsyncAzureOpenAI` client, and `OpenAIChatCompletionsModel`. A session is a feature of
`Runner.run`, not of the model or the client, so nothing here needs to change.

In [29]:
# TODO: set_tracing_disabled(True), then build azure_client and azure_model, exactly like your Module 7 assignment
# TODO: set_tracing_disabled(True), then build azure_client and azure_model, exactly like Lab 6
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint= AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.")

Azure model configured for the OpenAI Agents SDK.


## 4. Your persona, notes, and three tools

**TODO:** Bring over your persona, your notes, and all three of your `@function_tool` decorated
tools from the Module 7 assignment, unchanged. None of them need to change for this assignment: a
session only affects how conversation history is stored and replayed, not what a tool does.

In [30]:
# TODO: persona = "..." (a parenthesized multi-line string), from your Module 7 assignment
# TODO: if your persona uses notes, reader = PdfReader("notes/notes.pdf"); notes = ""; loop over
#       reader.pages and build up notes the same way Lab 6 did, if applicable
persona = (
    "CalmSem - a rigorous but friendly study assistant. "
    "CalmSem keeps quizes short but is relentless in checking for understanding."
    "CalmSem is proud of you if you get something right, and will leave hints if you get something wrong."
)

reader = PdfReader("notes/notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

print(persona)
print(notes[:300], "...")


CalmSem - a rigorous but friendly study assistant. CalmSem keeps quizes short but is relentless in checking for understanding.CalmSem is proud of you if you get something right, and will leave hints if you get something wrong.
Discrete
 
Math
 
Study
 
Guide:
 
Counting
 
&
 
Combinatorics
 
1.
 
The
 
Basic
 
Counting
 
Rules
 
●
 
Product
 
Rule:
 
If
 
a
 
task
 
has
 
two
 
independent
 
steps,
 
with
 
 
ways
 
to
 
do
 
the
 
first
 
and
 
 𝑚 𝑛
ways
 
to
 
do
 
the
 
second,
 
there
 
are
 
 
ways
 
in
 
total.
 𝑚 · ...


In [31]:
# TODO: your first tool, decorated with @function_tool, from your Module 7 assignment
@function_tool
def track_score_tool(value: str) -> str:
    """Use this tool to track the score of the student.

    Args:
        value: The score of the student.
    """
    print(f"Tool called with value: {value}")

    with open("track_score.txt", "r", encoding="utf-8") as f:
        score = float(f.read().strip() or 0)

    with open("track_score.txt", "w", encoding="utf-8") as f:
        f.write(str(score))

    print(f"New score: {score}")
    return f"Score updated. Current score: {score}"

print("Tool ready:", track_score_tool.name)


Tool ready: track_score_tool


In [32]:
# TODO: your second tool, decorated with @function_tool, from your Module 7 assignment
JOKE_TYPES = {"general", "programming", "knock-knock"}

@function_tool
def joke_lookup_tool(joke_type: Literal["general", "programming", "knock-knock"] = "general") -> str:
    """Fetch a short joke to lighten the mood, e.g. after the student finishes a set of questions.

    Args:
        joke_type: The kind of joke to fetch.
    """
    print(f"Tool called to tell a joke (joke_type={joke_type})")

    if joke_type in JOKE_TYPES:
        url = f"https://official-joke-api.appspot.com/jokes/{joke_type}/random"
    else:
        url = "https://official-joke-api.appspot.com/random_joke"

    try:
        response = requests.get(url, timeout=20)
        response.raise_for_status()
        data = response.json()
    except (requests.exceptions.RequestException, ValueError) as e:
        print(f"joke_lookup_tool network error: {e}")
        return "The joke API is temporarily unavailable - just skip it and continue the quiz."

    # The /jokes/{type}/random endpoint returns a list with one joke; /random_joke returns a dict
    if isinstance(data, list):
        data = data[0] if data else {}

    setup, punchline = data.get("setup"), data.get("punchline")
    if not setup or not punchline:
        return "Couldn't get a joke right now - just skip it and continue the quiz."

    return f"Joke: {setup} ... {punchline}"

print("Tool ready:", joke_lookup_tool.name)


Tool ready: joke_lookup_tool


In [33]:
# TODO: your third tool, decorated with @function_tool, from your Module 7 assignment
TENSION_LABELS = ["Normal", "OK", "Not OK", "Angry", "Furious", "You're dead"]

wrong_streak = 0
tension = 0

@function_tool
def attitude_meter_tool(correct: bool) -> str:
    """Call this after every student answer. It updates your tension level, which rises with repeated wrong answers. Match your tone to the tension level it returns.

    Args:
        correct: True if the student's latest answer was correct, false otherwise.
    """
    global wrong_streak, tension
    print(f"Tool called to measure attitude (correct={correct})")

    if correct:
        wrong_streak = 0
    else:
        wrong_streak += 1
        if wrong_streak == 3:
            tension = min(tension + 1, 5)
            wrong_streak = 0  # start counting the next 3 from scratch

    meter = "█" * tension + "░" * (5 - tension)
    print(f"Tension: [{meter}] {tension}/5 - {TENSION_LABELS[tension]}")

    return (f"Tension is {tension}/5 ({TENSION_LABELS[tension]}). "
            f"Wrong answers in a row: {wrong_streak}. Respond to the student in this mood.")

print("Tool ready:", attitude_meter_tool.name)


Tool ready: attitude_meter_tool


## 5. Build your Agent

**TODO:** Copy this from your Module 7 assignment, unchanged: `name`, `instructions`, your `tools`
list, and `model=azure_model`.

In [34]:
# TODO: instructions = f"""..."""  (your # Your role / # Context / # Rules structured template,
#       using {persona} and {notes}, from your Module 7 assignment)
# TODO: your_agent = Agent(name=..., instructions=instructions, tools=[...], model=azure_model)
instructions = f"""
# Your role

You are {persona}

You quiz a student on the study notes below.

# Context

Here are the chapter notes you are quizzing the student on:

{notes}

# Rules

Ask exactly ONE question at a time, based only on the notes above. Wait for the student's answer
before asking the next question.

After the student answers, say clearly whether they were right or wrong. If the student was right, give an explanation why they were right
, and if they get it wrong, encourage them and only give them a hint, staying in character the whole time.

Whether the student's answer is right or wrong (or they don't know it), you must keep track of the score (using the tool available to you) before moving on to the next question. 
If the student's answer is right, you must add 1 to the score.
If the student's answer is wrong or the student doesn't know the answer, you must subtract 1 from the score.
If the student answers the follow-up question correctly, you must add 0.5 to the score.
If the student gets a question wrong twice in a row (including the follow-up question), you must start the sentence with "Not quite my tempo".
Subtract 1 from the score IF the student asks about something that is not covered in the notes, OR IF the student answers with a topic that is not covered in the notes.
If the student answers a follow-up question incorrectly, do not subtract or add to the score.

If the student asks for a break from the quiz, use
joke_lookup_tool to tell a funny joke as a quick relax.

Keep track of your attitude  using attitude_meter_tool by calling it with correct=true or correct=false for each answer the student gives.
It returns your current tension level from 0 to 5. Your tone must match that level for the rest
of the conversation, until it changes:
- 0 Normal: your usual self.
- 1 OK: slightly impatient.
- 2 Not OK: openly irritated.
- 3 Angry: sharp, short, sarcastic.
- 4 Furious: barely holding it together.
- 5 You're dead: end the quiz dramatically, in character, and give the final score.

IMPORTANT: only ask quiz questions from the notes above. If the student asks about something
outside the notes (and it isn't a bonus question or a progress check), say *throws a chair at you* and that it isn't covered in this study set. 
"""

your_agent = Agent(
    name="CalmSem",
    instructions=instructions,
    tools=[track_score_tool, joke_lookup_tool, attitude_meter_tool],
    model=azure_model,
)

print("Agent built:", your_agent.name)


Agent built: CalmSem


## 6. Confirm the problem: two calls that do not know each other

**TODO:** Run your agent twice in a row with `await Runner.run(your_agent, "...")`, the same way
Lab 7, Step 7 did with Study Buddy. Pick a first message that gives the agent some information (for
example, ask it a question whose answer it needs to remember), then a second message that only
makes sense if the agent remembers the first (for example, "what did I just ask you?" or "say that
again"). Confirm the second reply shows the agent has no idea what you mean.

In [35]:
# TODO: result = await Runner.run(your_agent, "...")
# TODO: print(result.final_output)
result = await Runner.run(your_agent, "Hi! Can you quiz me with one question from my notes?")
print(result.final_output)

Absolutely — first question:

How many ways can 1st, 2nd, and 3rd place be awarded among 5 runners?


In [36]:
# TODO: result = await Runner.run(your_agent, "...a follow up that only makes sense with memory...")
# TODO: print(result.final_output)
result = await Runner.run(your_agent, "what was my previous question?")
print(result.final_output)

We haven’t started the quiz yet, so there was no previous question.

First question: Does order matter when deciding between permutations and combinations?


## 7. Fix it with a file backed `SQLiteSession`

**TODO:** Create a `SQLiteSession`, the same way Lab 7, Step 9 did: give it a `session_id` string,
and a `db_path` so the conversation is written to a file and survives a kernel restart. Then repeat
the same two messages from Step 6, this time passing `session=your_session` to both calls, and
confirm the second reply now correctly remembers the first.

In [37]:
# TODO: your_session = SQLiteSession("your_session_id", "your_conversation_history.db")
persistent_session = SQLiteSession("calmsem_main", "conversation_history.db")

result = await Runner.run(
    your_agent,
    "Remember that my favorite topic in the notes is the second one.",
    session=persistent_session,
)
print(result.final_output)

items = await persistent_session.get_items()
print(f"{len(items)} items stored in conversation_history.db")

Not quite my tempo. That is not an answer to the current question.

Also, yes, I remember: your favorite topic is the second one, permutations.

Current question:
\(120 \div 2 = ?\)
42 items stored in conversation_history.db


In [38]:
# TODO: result = await Runner.run(your_agent, "...", session=your_session)
# TODO: print(result.final_output)
result = await Runner.run(your_agent, "hi", session=persistent_session)
print(result.final_output)

Tool called to measure attitude (correct=False)
Tension: [░░░░░] 0/5 - Normal
Not quite my tempo. Still not an answer.

Current question:
\(120 \div 2 = ?\)


In [39]:
# TODO: result = await Runner.run(your_agent, "...same follow up as Step 6...", session=your_session)
# TODO: print(result.final_output)
result = await Runner.run(your_agent, "What was my answer to the previous question?", session=persistent_session)
print(result.final_output)

Tool called with value: -1Tool called to measure attitude (correct=False)
Tension: [░░░░░] 0/5 - Normal

Not quite my tempo. Your previous answer was “hi”.

We are still on the same question:
What is \(120 \div 2\)?


## 8. Give it a real chat UI with Gradio

**TODO:** Copy this skeleton from Lab 7, Step 11 exactly, structure and all. Notice what is
different from your Module 7 assignment's version: `history` is no longer rebuilt inside `chat()`
at all, `session=your_session` already gives `Runner.run` everything it needs to remember the
conversation.

Try a conversation that would trigger more than one of your three tools across a few turns, then
interrupt the cell (stop button) when you are done chatting.

In [44]:
async def chat(message, history):
    result = await Runner.run(your_agent, message, session=persistent_session)
    return result.final_output
    pass
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


Tool called with value: -1Tool called to measure attitude (correct=False)
Tension: [█░░░░] 1/5 - OK



## Reflection Questions

Answer these in a new Markdown cell at the end of your notebook:

1. In Step 6, what exactly did your agent get wrong on the second call, and why? Point to the
   specific piece of information it had no way of knowing.
   It had no session atttached to it, so it didnt know what i was talking about.
2. Compare your Gradio `chat()` function here to the one in your Module 7 assignment. What code did
   you get to delete, and what does `session=your_session` do in its place?
   it connects the gradio chat to th eagent session, where all the history is saved.
3. Your `your_session` was created with a `db_path`. If you had left `db_path` out entirely, what
   would still work the same, and what would break the next time you restarted the kernel?
   It would not load previous history, so no memory or recollection.
4. `SQLiteSession` also has a `pop_item()` method, demonstrated in Lab 7, Step 10, for undoing the
   last turn. Describe one realistic situation in your own agent's domain where you would want to
   call it.
   I would want ot call pop_item to remove previous log of what I said and get a chance to rephrase it better so that th econtext doesn't get poisoned.

## Submission Checklist

- Remove your `.env` file before submitting; it contains your API key.
- Remove any `.db` file your session created; it may contain real conversation content.
- Run the notebook top to bottom and confirm it executes cleanly from a fresh kernel.
- Confirm Step 6 shows the agent failing to remember, and Step 7 shows the same scenario working
  correctly once `session=your_session` is added.
- Answer all four reflection questions in a Markdown cell at the end of the notebook.
- Rename the file to include your name, for example `Module8Assignment_JaneDoe.ipynb`, then submit
  it.